# 3.11 Python

Python organises its named probability distributions under `scipy.stats`. Each distribution is an object that exposes a uniform interface: `.pmf` for the probability mass function, `.cdf` for the cumulative distribution function, and so on. NumPy's random generator handles simulation. This section works through the Binomial and Hypergeometric distributions from this chapter, then shows how to draw samples from an arbitrary discrete distribution defined by its support and probability weights.

In [ ]:
import numpy as np
import scipy.stats as st
import seaborn as sns
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

## Binomial Distribution

If $X \sim \mathrm{Bin}(n, p)$, the PMF is

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}, \quad k = 0, 1, \ldots, n.$$

Three functions cover the most common tasks:

- `st.binom.pmf(k, n, p)` — PMF evaluated at $k$
- `st.binom.cdf(k, n, p)` — CDF evaluated at $k$, i.e. $P(X \le k)$
- `rng.binomial(n, p, size=N)` — $N$ independent draws

For example, with $X \sim \mathrm{Bin}(5, 0.2)$:

In [ ]:
n, p = 5, 0.2
st.binom.pmf(3, n, p)   # P(X = 3)

This equals $\binom{5}{3}(0.2)^3(0.8)^2 = 0.0512$. The CDF sums the PMF from 0 up through the given value:

In [ ]:
st.binom.cdf(3, n, p)   # P(X <= 3)

`rng.binomial(n, p, size=7)` produces seven independent realisations of $\mathrm{Bin}(5, 0.2)$ — each value is a count of successes in 5 Bernoulli trials:

In [ ]:
rng.binomial(n, p, size=7)

Because `scipy.stats` functions accept NumPy arrays, passing `np.arange(0, n+1)` as the first argument returns the complete PMF — all six probabilities at once:

In [ ]:
k = np.arange(0, n + 1)
pmf = st.binom.pmf(k, n, p)
pmf

A bar chart is the natural display for a discrete PMF:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=k, y=pmf, color="steelblue", ax=ax)
ax.set_xlabel("k")
ax.set_ylabel("P(X = k)")
ax.set_title(f"Binomial PMF  (n = {n}, p = {p})")
plt.tight_layout()
plt.show()

## Hypergeometric Distribution

The Hypergeometric distribution models sampling *without replacement* from a population of $w$ white and $b$ black balls, drawing $n$ at random. Writing $X \sim \mathrm{HGeom}(w, b, n)$, the PMF is

$$P(X = k) = \frac{\dbinom{w}{k}\dbinom{b}{n-k}}{\dbinom{w+b}{n}}.$$

`scipy.stats.hypergeom` uses the parameterisation `hypergeom(M, n_good, N)` where $M = w + b$ is the population size, $n_{\text{good}} = w$ is the number of white balls, and $N = n$ is the number drawn. In terms of the textbook's $\mathrm{HGeom}(w, b, n)$ notation:

```
st.hypergeom.pmf(k, w + b, w, n)   # P(X = k)
st.hypergeom.cdf(k, w + b, w, n)   # P(X <= k)
```

As a concrete example, take $w = 10$, $b = 5$, $n = 8$:

In [ ]:
w, b, n_draws = 10, 5, 8

print(st.hypergeom.pmf(3, w + b, w, n_draws))   # P(X = 3)
print(st.hypergeom.cdf(3, w + b, w, n_draws))   # P(X <= 3)

The possible values of $X$ range from $\max(0,\, n - b)$ to $\min(w,\, n)$. With 10 white and 5 black balls and 8 draws, we must pick at least 3 white balls (since there are only 5 black ones), so the PMF is zero outside that range:

In [ ]:
k_lo = max(0, n_draws - b)
k_hi = min(w, n_draws)
k_h = np.arange(k_lo, k_hi + 1)
pmf_h = st.hypergeom.pmf(k_h, w + b, w, n_draws)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=k_h, y=pmf_h, color="slateblue", ax=ax)
ax.set_xlabel("k  (white balls drawn)")
ax.set_ylabel("P(X = k)")
ax.set_title(f"Hypergeometric PMF  (w={w}, b={b}, n={n_draws})")
plt.tight_layout()
plt.show()

`rng.hypergeometric(ngood, nbad, nsample, size=N)` generates $N$ independent realisations. The three non-`size` arguments are the number of white balls, the number of black balls, and the number drawn — matching the textbook's $w$, $b$, $n$:

In [ ]:
rng.hypergeometric(w, b, n_draws, size=10)

## Discrete Distributions with Finite Support

For a distribution not built into `scipy.stats`, we can simulate directly using `rng.choice`. The key is specifying the support and the corresponding probabilities via the `p` argument.

Suppose $X$ has the PMF

$$P(X = 0) = 0.25,\quad P(X = 1) = 0.50,\quad P(X = 5) = 0.10,\quad P(X = 10) = 0.15.$$

We encode the support and probabilities as arrays, then pass them to `rng.choice`:

In [ ]:
x = np.array([0, 1, 5, 10])
p = np.array([0.25, 0.50, 0.10, 0.15])

rng.choice(x, size=100, p=p)   # 100 draws from this PMF

The probabilities in `p` must sum to 1; NumPy will raise an error if they do not. We can verify the simulated frequencies against the stated PMF by tallying the draws:

In [ ]:
draws = rng.choice(x, size=10**5, p=p)
for val, prob in zip(x, p):
    simulated = np.mean(draws == val)
    print(f"P(X = {val:2d})  stated: {prob:.2f}  simulated: {simulated:.4f}")

With 100 000 draws the simulated frequencies are very close to the stated probabilities. We can also plot the empirical distribution alongside the exact PMF:

In [ ]:
simulated_probs = np.array([np.mean(draws == val) for val in x])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
sns.barplot(x=x, y=p,               color="seagreen", ax=axes[0])
sns.barplot(x=x, y=simulated_probs, color="coral",    ax=axes[1])
for ax, title in zip(axes, ["Stated PMF", "Simulated (n = 100 000)"]):
    ax.set_xlabel("x")
    ax.set_ylabel("Probability")
    ax.set_title(title)
plt.tight_layout()
plt.show()